In [ ]:
# FINAL OPTIMIZED PIPELINE (DYNAMIC PARALLEL + CHECKPOINT + PROGRESS + CLEAN OUTPUT)

import os
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.ensemble import IsolationForest, RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier, XGBRegressor
from joblib import Parallel, delayed
from tqdm import tqdm
import torch
import torch.nn as nn
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

import category_encoders as ce


############################
# SILENCE WARNINGS
############################
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

############################
# CONFIG
############################
DATA_PATH = "data"
OUTPUT_PATH = "results_cv"
COMBO_PATH = os.path.join(OUTPUT_PATH, "per_combo")
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(COMBO_PATH, exist_ok=True)

N_SPLITS = 10
N_REPEATS = 3
N_JOBS = 12

DATASETS = [
"Bank_Customer_Churn_Dataset","cmc","connect-4","credit-g",
"elevators","eucalyptus","eye_movements","house_16H","kc1",
"phoneme","pol","socmob","splice","vehicle",
"california_housing","cpu_act","Bike_Sharing_Demand","abalone","wine_quality"
]

# ✅ AGGIUNTO mlp_deep
MODELS = ["mlp","mlp_deep","xgboost","randomforest"]

ENCODINGS = [
    "ordinal",
    "target",
    "catboost"
]

MISSING_METHODS = [
    "mean",
    "mean_indicator",
    "mice"
]

############################
# DATA
############################

def load_json(path):
    return json.loads(Path(path).read_text())

def load_dataset(dataset):
    base = Path(DATA_PATH)/dataset
    def block(name):
        return {x: np.load(base/f"{name}_{x}.npy", allow_pickle=True) for x in ["train","test"]}
    N = block("N") if (base/"N_train.npy").exists() else None
    C = block("C") if (base/"C_train.npy").exists() else None
    y = block("y")
    info = load_json(base/"info.json")
    return N,C,y,info

def apply_encoding(
        X_train,
        X_test,
        y_train,
        method):

    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)

    cat_cols = X_train.select_dtypes(
        include=["object"]
    ).columns

    if len(cat_cols) == 0:
        return X_train.values, X_test.values

    if method == "ordinal":

        for col in cat_cols:

            le = LabelEncoder()

            X_train[col] = le.fit_transform(
                X_train[col].astype(str)
            )

            mapping = {
                c:i
                for i,c in enumerate(le.classes_)
            }

            X_test[col] = (
                X_test[col]
                .astype(str)
                .map(mapping)
                .fillna(0)
            )

        return X_train.values, X_test.values

    if method == "target":

        enc = ce.TargetEncoder()

        X_train = enc.fit_transform(
            X_train,
            y_train
        )

        X_test = enc.transform(
            X_test
        )

        return X_train.values, X_test.values

    if method == "catboost":

        enc = ce.CatBoostEncoder()

        X_train = enc.fit_transform(
            X_train,
            y_train
        )

        X_test = enc.transform(
            X_test
        )

        return X_train.values, X_test.values

    return X_train.values, X_test.values


def apply_missing_strategy(
        X_train,
        X_test,
        method):

    if method == "mean":

        imp = SimpleImputer(
            strategy="mean"
        )

    elif method == "mean_indicator":

        imp = SimpleImputer(
            strategy="mean",
            add_indicator=True
        )

    elif method == "mice":

        imp = IterativeImputer(
            random_state=42
        )

    X_train = imp.fit_transform(
        X_train
    )

    X_test = imp.transform(
        X_test
    )

    return X_train, X_test

############################
# ENCODING
############################

def encode_train_test(X_train, X_test):
    X_train = pd.DataFrame(X_train)
    X_test = pd.DataFrame(X_test)
    for col in X_train.columns:
        if X_train[col].dtype == object:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col].astype(str))
            X_test[col] = X_test[col].astype(str).map(lambda x: le.transform([x])[0] if x in le.classes_ else 0)
    return X_train.values, X_test.values

############################
# PREPROCESS
############################

def preprocess_train(X):
    X = pd.DataFrame(X)
    means, stds = {}, {}
    for col in X.columns:
        if np.issubdtype(X[col].dtype, np.number):
            m = X[col].mean()
            s = X[col].std() or 1
            X[col] = (X[col].fillna(m) - m) / s
            means[col], stds[col] = m, s
    return X.fillna(0).values, means, stds


def preprocess_test(X, means, stds):
    X = pd.DataFrame(X)
    for col in means:
        if col in X:
            X[col] = (X[col].fillna(means[col]) - means[col]) / stds[col]
    return X.fillna(0).values

############################
# AUTOENCODER
############################

class AE(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d,64), nn.ReLU(), nn.Linear(64,16))
        self.decoder = nn.Sequential(nn.Linear(16,64), nn.ReLU(), nn.Linear(64,d))
    def forward(self,x): return self.decoder(self.encoder(x))


def ae_outliers(X, seed):
    torch.manual_seed(seed)
    X_t = torch.tensor(X, dtype=torch.float32)
    model = AE(X.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    for _ in range(20):
        out = model(X_t); loss = loss_fn(out, X_t)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        err = ((X_t - model(X_t))**2).mean(1).numpy()
    thresh = np.percentile(err, 100*(1-CONTAMINATION))
    return err < thresh, None, None

############################
# OUTLIERS
############################

def detect_outliers(X, method, seed):
    X = np.nan_to_num(np.asarray(X).astype(float))
    if method is None: return np.ones(len(X), bool), None, None
    if method == "ZScore":
        mean,std = X.mean(0), X.std(0); std[std==0]=1
        lower,upper = mean-3*std, mean+3*std
        return ((X>=lower)&(X<=upper)).all(1), lower, upper
    if method == "Percentile":
        lower = np.percentile(X,LOW_PCTL,axis=0); upper = np.percentile(X,HIGH_PCTL,axis=0)
        return ((X>=lower)&(X<=upper)).all(1), lower, upper
    if method == "IsolationForest":
        return IsolationForest(contamination=CONTAMINATION,random_state=seed).fit_predict(X)==1,None,None
    if method == "LOF":
        return LocalOutlierFactor(contamination=CONTAMINATION).fit_predict(X)==1,None,None
    if method == "Autoencoder": return ae_outliers(X,seed)
    return np.ones(len(X), bool), None, None


def clip_numeric(X, lower, upper):
    return X if lower is None else np.clip(X,lower,upper)

############################
# MODELS
############################

def make_model(name, task, seed):

    if name == "mlp":
        if task == "classification":
            return MLPClassifier(hidden_layer_sizes=(100,), max_iter=300,
                                 early_stopping=True, n_iter_no_change=10,
                                 random_state=seed)
        return MLPRegressor(hidden_layer_sizes=(100,), max_iter=300,
                            early_stopping=True, n_iter_no_change=10,
                            random_state=seed)

    if name == "mlp_deep":
        if task == "classification":
            return MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=300,
                                 early_stopping=True, n_iter_no_change=10,
                                 random_state=seed)
        return MLPRegressor(hidden_layer_sizes=(128,64,32), max_iter=300,
                            early_stopping=True, n_iter_no_change=10,
                            random_state=seed)

    if name == "xgboost":
        if task == "classification":
            return XGBClassifier(n_estimators=200, max_depth=6,
                                 n_jobs=1, verbosity=0,
                                 eval_metric="logloss")
        return XGBRegressor(n_estimators=200, max_depth=6,
                            n_jobs=1, verbosity=0)

    if name == "randomforest":
        if task == "classification":
            return RandomForestClassifier(n_estimators=100,
                                          n_jobs=1,
                                          random_state=seed)
        return RandomForestRegressor(n_estimators=100,
                                     n_jobs=1,
                                     random_state=seed)

############################
# CORE
############################

def generate_splits(n):
    splits=[]
    for r in range(N_REPEATS):
        kf=KFold(n_splits=N_SPLITS,shuffle=True,random_state=42+r)
        for f,(tr,te) in enumerate(kf.split(range(n))):
            splits.append((r,f,tr,te))
    return splits


def combo_filename(
        dataset,
        model,
        encoding,
        missing_method):

    return os.path.join(
        COMBO_PATH,
        f"{dataset}__{model}__{encoding}__{missing_method}.csv"
    )


def process_combo(
        dataset,
        model,
        encoding,
        missing_method):

    out_file = combo_filename(dataset, model, detector, treatment)
    if os.path.exists(out_file):
        return pd.read_csv(out_file)

    N,C,y,info = load_dataset(dataset)
    y_all = y["train"].ravel()
    task = "classification" if info["task_type"]!="regression" else "regression"

    X_all = np.hstack([N["train"],C["train"]]) if N is not None and C is not None else (N["train"] if N is not None else C["train"])
    splits = generate_splits(len(X_all))

    rows=[]
    for run,fold,tr_idx,te_idx in splits:

        X_train,X_test = X_all[tr_idx],X_all[te_idx]
        y_train,y_test = y_all[tr_idx],y_all[te_idx]

        if task=="classification":
            le=LabelEncoder()
            y_train=le.fit_transform(y_train)
            y_test=np.array([le.transform([x])[0] if x in le.classes_ else 0 for x in y_test])

        X_train,X_test = apply_encoding(
            X_train,
            X_test,
            y_train,
            encoding
        )
        
        X_train,X_test = apply_missing_strategy(
            X_train,
            X_test,
            missing_method
        )
        
        X_train,means,stds = preprocess_train(X_train)
        X_test = preprocess_test(X_test,means,stds)

        model_obj = make_model(model,task,seed=run*100+fold)
        model_obj.fit(X_train,y_train)
        pred = model_obj.predict(X_test)

        score = accuracy_score(y_test,pred) if task=="classification" else mean_squared_error(y_test,pred)

        rows.append({
            "dataset":dataset,
            "model":model,
            "encoding":encoding,
            "missing":missing_method,
            "preprocessing":f"{encoding}_{missing_method}",
            "run":run,
            "fold":fold,
            "score":score
        })

    df=pd.DataFrame(rows)
    df.to_csv(out_file,index=False)
    return df

############################
# RUN
############################



def run_all():

    combos = [

        (d,m,e,ms)

        for d in DATASETS
        for m in MODELS
        for e in ENCODINGS
        for ms in MISSING_METHODS
    ]

    remaining = [
        c for c in combos
        if not os.path.exists(
            combo_filename(*c)
        )
    ]

    print(
        f"Remaining combos: {len(remaining)}"
    )

    results = []

    with Parallel(
        n_jobs=N_JOBS,
        return_as="generator"
    ) as parallel:

        tasks = (
            delayed(process_combo)(
                c[0],
                c[1],
                c[2],
                c[3]
            )
            for c in remaining
        )

        for res in tqdm(
            parallel(tasks),
            total=len(remaining),
            desc="Progress"
        ):
            results.append(res)

    final_df = pd.concat(
        results,
        ignore_index=True
    )

    final_df.to_csv(
        f"{OUTPUT_PATH}/all_results.csv",
        index=False
    )

    return final_df


if __name__=="__main__":
    df=run_all()
    print(df.head())